# skforecast-ai — Demo Phase 5

This notebook demonstrates **Phase 5: CLI (Tier 0 Complete)**.

The CLI exposes the full deterministic pipeline (`inspect`, `recommend`,
`generate-code`) as shell commands. No LLM, no API key, no network required.

We create a sample CSV and exercise all three commands with their key options.

## Setup — Create a sample CSV

In [1]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

# Create a realistic daily time series with an exogenous variable
rng = np.random.default_rng(42)
dates = pd.date_range("2023-01-01", periods=365, freq="D")
df = pd.DataFrame(
    {
        "sales": rng.normal(200, 25, 365).round(2),
        "temperature": rng.normal(18, 8, 365).round(1),
    },
    index=dates,
)
df.index.name = "date"

# Write to a temporary CSV
tmp_dir = Path(tempfile.mkdtemp())
csv_path = tmp_dir / "daily_sales.csv"
df.to_csv(csv_path)

print(f"CSV written to: {csv_path}")
print(f"Shape: {df.shape}")
df.head()

CSV written to: /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/tmpvi389mtt/daily_sales.csv
Shape: (365, 2)


,sales,temperature
date,,
2023-01-01,207.62,29.9
2023-01-02,174.00,12.1
2023-01-03,218.76,11.4
2023-01-04,223.51,19.6
2023-01-05,151.22,24.8


## 1. `skforecast-ai inspect` — Profile the dataset

The `inspect` command loads the CSV, detects structure and frequency,
and prints a `DataProfile`. Use `--json` for machine-readable output.

In [2]:
!skforecast-ai inspect {csv_path} --target sales --json

{
  "n_observations": 365,
  "n_series": 1,
  "index_type": "datetime",
  "frequency": "D",
  "target": "sales",
  "date_column": null,
  "series_id_column": null,
  "exog_columns": [
    "temperature"
  ],
  "categorical_exog": [],
  "missing_values": {},
  "inferred_seasonalities": [
    7,
    365
  ],
  "warnings": []
}


In [3]:
# Human-readable output (Rich table)
!skforecast-ai inspect {csv_path} --target sales

           Data Profile           
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Field            ┃ Value       ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ Target           │ sales       │
│ Observations     │ 365         │
│ Series           │ 1           │
│ Index type       │ datetime    │
│ Frequency        │ D           │
│ Date column      │ —           │
│ Series ID column │ —           │
│ Exog columns     │ temperature │
│ Categorical exog │ None        │
│ Seasonalities    │ [7, 365]    │
│ Missing values   │ None        │
└──────────────────┴─────────────┘


## 2. `skforecast-ai recommend` — Generate a forecasting plan

The `recommend` command profiles the dataset and then applies the
deterministic rule engine to select forecaster, estimator, lags, metric, etc.

In [4]:
!skforecast-ai recommend {csv_path} --target sales --horizon 30 --json

{
  "task_type": "single_series",
  "forecaster": "ForecasterRecursive",
  "estimator": "Ridge",
  "horizon": 30,
  "frequency": "D",
  "lags": [
    1,
    2,
    3,
    4,
    5,
    6,
    7
  ],
  "metric": "mean_absolute_error",
  "backtesting_strategy": "TimeSeriesFold",
  "interval_method": "bootstrapping",
  "use_exog": true,
  "data_requirements": [],
  "warnings": [],
  "rationale": "A single-series ML forecaster (ForecasterRecursive) is recommended. The estimator is Ridge. Lags: [1, 2, 3, 4, 5, 6, 7]. Metric: mean_absolute_error. Prediction intervals via bootstrapping. Exogenous variables detected: ['temperature']."
}


In [5]:
# Human-readable output
!skforecast-ai recommend {csv_path} --target sales --horizon 30

                                 Forecast Plan                                  
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Field           ┃ Value                                                      ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Task type       │ single_series                                              │
│ Forecaster      │ ForecasterRecursive                                        │
│ Estimator       │ Ridge                                                      │
│ Horizon         │ 30                                                         │
│ Frequency       │ D                                                          │
│ Lags            │ [1, 2, 3, 4, 5, 6, 7]                                      │
│ Metric          │ mean_absolute_error                                        │
│ Backtesting     │ TimeSeriesFold                                             │
│ Interval method │ bootstra

## 3. `skforecast-ai generate-code` — Produce a ready-to-run script

The `generate-code` command runs the full pipeline (profile → recommend → generate)
and outputs a self-contained Python script.

In [6]:
# Print code to stdout
!skforecast-ai generate-code {csv_path} --target sales --horizon 30

import pandas as pd
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import backtesting_forecaster, TimeSeriesFold
from sklearn.linear_model import Ridge

# Load data
data = pd.read_csv('/var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/tmpvi389mtt/daily_sales.csv', index_col=0, parse_dates=True)
data = data.asfreq('D')

# Train/test split (time-based)
n_train = int(len(data) * 0.8)
y_train = data.iloc[:n_train]['sales']
y_test = data.iloc[n_train:]['sales']
exog_train = data.iloc[:n_train][['temperature']]
exog_test = data.iloc[n_train:][['temperature']]

# Create forecaster
forecaster = ForecasterRecursive(
    estimator = Ridge(random_state=123),
    lags      = [1, 2, 3, 4, 5, 6, 7],
)

# Fit
forecaster.fit(y=y_train, exog=exog_train)

# Predict
predictions = forecaster.predict(steps=30, exog=exog_test.iloc[:30])
print(predictions)

# Backtesting
cv = TimeSeriesFold(
    steps              = 30,
    initial_train_size = n_train,
    refit         

In [7]:
# Write code to a file
output_script = tmp_dir / "forecast_script.py"
!skforecast-ai generate-code {csv_path} --target sales --horizon 30 --output {output_script}

print(f"\nFile exists: {output_script.exists()}")
print(f"File size: {output_script.stat().st_size} bytes")

Code written to: 
/var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/tmpvi389mtt/forecast_script.py

File exists: True
File size: 1551 bytes


In [8]:
# Verify the generated script is syntactically valid
script_content = output_script.read_text()
compile(script_content, "<forecast_script>", "exec")
print("Syntax check: OK")

Syntax check: OK


## 4. `--json` flag on `generate-code`

Returns both the plan and the generated code in a single JSON object.

In [9]:
import json
import subprocess

result = subprocess.run(
    ["skforecast-ai", "generate-code", str(csv_path),
     "--target", "sales", "--horizon", "30", "--json"],
    capture_output=True, text=True,
)

output = json.loads(result.stdout)
print(f"Keys: {list(output.keys())}")
print(f"Plan task_type: {output['plan']['task_type']}")
print(f"Plan forecaster: {output['plan']['forecaster']}")
print(f"Code length: {len(output['code'])} characters")

Keys: ['plan', 'code']
Plan task_type: single_series
Plan forecaster: ForecasterRecursive
Code length: 1551 characters


## 5. Error handling

The CLI provides clear error messages without stack traces.

In [10]:
# Nonexistent file → exit code 1
!skforecast-ai inspect nonexistent.csv --target sales; echo "Exit code: $?"

Error: File not found: nonexistent.csv
Exit code: 1


In [11]:
# Missing required option
!skforecast-ai inspect {csv_path}; echo "Exit code: $?"

Usage: skforecast-ai inspect [OPTIONS] DATA_PATH
Try 'skforecast-ai inspect --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ Missing option '--target'.                                                   │
╰──────────────────────────────────────────────────────────────────────────────╯
Exit code: 2


## 6. Multi-series example

The CLI also supports multi-series datasets via `--series-id`.

In [12]:
# Create a multi-series CSV
rng = np.random.default_rng(99)
dates_multi = pd.date_range("2023-01-01", periods=200, freq="D")
df_multi = pd.DataFrame({
    "date": np.tile(dates_multi, 3),
    "store": np.repeat(["A", "B", "C"], 200),
    "revenue": rng.normal(1000, 100, 600).round(2),
})

csv_multi = tmp_dir / "multi_stores.csv"
df_multi.to_csv(csv_multi, index=False)

print(f"Multi-series CSV: {csv_multi}")
print(f"Shape: {df_multi.shape}")

Multi-series CSV: /var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/tmpvi389mtt/multi_stores.csv
Shape: (600, 3)


In [13]:
!skforecast-ai recommend {csv_multi} --target revenue --date date --series-id store --horizon 14 --json

{
  "task_type": "multi_series",
  "forecaster": "ForecasterRecursiveMultiSeries",
  "estimator": "LGBMRegressor",
  "horizon": 14,
  "frequency": null,
  "lags": [
    1,
    2,
    3,
    4,
    5,
    6,
    7
  ],
  "metric": "mean_absolute_error",
  "backtesting_strategy": "TimeSeriesFold",
  "interval_method": "conformal",
  "use_exog": false,
  "data_requirements": [
    "Provide a DatetimeIndex or date column for time-based features."
  ],
  "warnings": [],
  "rationale": "The dataset contains 3 series, so a multi-series forecaster (ForecasterRecursiveMultiSeries) is recommended. The estimator is LGBMRegressor. Lags: [1, 2, 3, 4, 5, 6, 7]. Metric: mean_absolute_error. Prediction intervals via conformal."
}


In [14]:
!skforecast-ai generate-code {csv_multi} --target revenue --date date --series-id store --horizon 14

import pandas as pd
from skforecast.recursive import ForecasterRecursiveMultiSeries
from skforecast.model_selection import backtesting_forecaster_multiseries, TimeSeriesFold
from lightgbm import LGBMRegressor

# Load data (long format)
data = pd.read_csv('/var/folders/wt/8tvn563d5v55nspfbydgqb9r0000gp/T/tmpvi389mtt/multi_stores.csv', parse_dates=['date'])

# Pivot to wide format (columns = series)
series = data.pivot_table(
    index='date', columns='store', values='revenue'
)
series.index = pd.DatetimeIndex(series.index)
series.index.name = None

# Train/test split
n_train = int(len(series) * 0.8)

# Create forecaster
forecaster = ForecasterRecursiveMultiSeries(
    estimator = LGBMRegressor(random_state=123),
    lags      = [1, 2, 3, 4, 5, 6, 7],
    encoding  = 'ordinal',
)

# Fit
forecaster.fit(series=series)

# Predict
predictions = forecaster.predict(steps=14)
print(predictions)

# Backtesting
cv = TimeSeriesFold(
    steps              = 14,
    initial_train_size = n_train,
  

## Summary

Phase 5 completes **Tier 0** — a fully usable forecasting assistant with:

- `skforecast-ai inspect` — data profiling
- `skforecast-ai recommend` — deterministic forecasting plan
- `skforecast-ai generate-code` — ready-to-run Python script

All deterministic, no LLM, no API key, no network required.